# Cache Keypoints

Run HRNet over every clip once, using GT person bboxes from `anns.json`, and dump 17 keypoints per person per frame to disk. The GNN/LSTM/MLP trainers all read from this cache instead of re-running pose estimation every epoch.

Output: `data/processed/CafeV1/cache/keypoints/<vp>/<clip>.json`

## Imports

In [4]:
import sys
import json
from collections import defaultdict
from pathlib import Path

import cv2

In [5]:
sys.path.insert(0, str(Path.cwd().parent))

In [6]:
from src.utils.config.config import Config
from src.pose_estimator.hrnet_pose import HrNetPose
from src.classifier.common import keypoint_cache

## Config

In [7]:
PERSON_CAT_ID = 1

config_loader = Config()
cfg = config_loader.load_config()

dataset_root = cfg.project_root / cfg.paths.dataset / "CafeV1"
clips_root   = dataset_root / "Clips"
cache_root   = dataset_root / "cache" / "keypoints"

hrnet = HrNetPose(cfg.hrnet.model_cfg, cfg.hrnet.model_ckpt, cfg.device)

Loads checkpoint by local backend from path: C:\Research\Prototype\skeleton-based-hoi-recognition\models\hrnet\td-hm_hrnet-w32_udp-8xb64-210e_coco-256x192-73ede547_20220914.pth


## Helpers

We use GT person bboxes from `anns.json` (not YOLO predictions) so the cached keypoints are as clean as the annotations. That way detector errors don't contaminate the action classifier comparison.

In [8]:
# CVAT exports track_id as int or str, sometimes empty so normalize to int or None
def parse_track_id(ann):
    tid = ann.get("attributes", {}).get("track_id")
    if tid is None or tid == "":
        return None
    try:
        return int(tid)
    except (TypeError, ValueError):
        return None

In [9]:
def load_clip_anns(anns_path):
    with open(anns_path, "r") as f:
        coco = json.load(f)

    images = {img["id"]: img["file_name"] for img in coco["images"]}

    persons_by_frame = defaultdict(list)
    for ann in coco.get("annotations", []):
        if ann["category_id"] != PERSON_CAT_ID:
            continue
        tid = parse_track_id(ann)
        if tid is None:
            continue
        x, y, w, h = ann["bbox"]
        persons_by_frame[ann["image_id"]].append((tid, [x, y, x + w, y + h]))

    return images, dict(persons_by_frame)

In [10]:
def infer_frame(frame, person_dets):
    if not person_dets:
        return {}

    track_ids = [d[0] for d in person_dets]
    bboxes    = [d[1] for d in person_dets]

    samples = hrnet.infer(frame, bboxes)

    out = {}
    for tid, s in zip(track_ids, samples):
        kpts   = s.pred_instances.keypoints[0]        # (17, 2)
        scores = s.pred_instances.keypoint_scores[0]  # (17,)
        out[tid] = [[float(x), float(y), float(c)] for (x, y), c in zip(kpts, scores)]
    return out

In [11]:
def process_clip(clip_dir):
    anns_path  = clip_dir / "anns.json"
    images_dir = clip_dir / "images"

    images, persons_by_frame = load_clip_anns(anns_path)

    frame_kpts = {}
    for fid in sorted(images):
        dets = persons_by_frame.get(fid, [])
        if not dets:
            continue

        frame = cv2.imread(str(images_dir / images[fid]))
        if frame is None:
            print(f"  could not read {images[fid]}")
            continue

        by_track = infer_frame(frame, dets)
        if by_track:
            frame_kpts[fid] = by_track

    return frame_kpts

## Run

In [12]:
viewpoints = sorted([p.name for p in clips_root.iterdir() if p.is_dir()], key=int)

total = 0
skipped = 0
for vp in viewpoints:
    clip_dirs = sorted(
        [p for p in (clips_root / vp).iterdir() if p.is_dir()],
        key=lambda p: int(p.name)
    )

    for clip_dir in clip_dirs:
        clip = clip_dir.name

        if not (clip_dir / "anns.json").exists():
            print(f"vp{vp}/{clip}: no anns.json, skipping")
            skipped += 1
            continue

        if keypoint_cache.exists(cache_root, vp, clip):
            print(f"vp{vp}/{clip}: cached")
            skipped += 1
            continue

        print(f"vp{vp}/{clip}...", end=" ", flush=True)
        frame_kpts = process_clip(clip_dir)
        keypoint_cache.save(cache_root, vp, clip, frame_kpts)
        print(f"{len(frame_kpts)} frames")
        total += 1

print(f"\nDone. Cached {total} clips, skipped {skipped}.")
print(f"Cache root: {cache_root}")

vp1/0... 30 frames
vp1/1... 32 frames
vp1/2... 32 frames
vp1/19... 34 frames
vp1/20... 19 frames
vp1/21... 32 frames
vp1/22... 32 frames
vp1/450... 32 frames
vp1/451... 33 frames
vp2/0... 30 frames
vp2/1... 32 frames
vp2/2... 32 frames
vp2/19... 34 frames
vp2/20... 31 frames
vp2/21... 32 frames
vp2/22... 32 frames
vp2/450... 32 frames
vp2/451... 33 frames
vp3/0... 30 frames
vp3/1... 32 frames
vp3/2... 32 frames
vp3/19... 34 frames
vp3/20... 31 frames
vp3/21... 32 frames
vp3/22... 32 frames
vp3/450... 32 frames
vp3/451... 33 frames
vp4/0... 30 frames
vp4/1... 32 frames
vp4/2... 32 frames
vp4/19... 34 frames
vp4/20... 31 frames
vp4/21... 32 frames
vp4/22... 32 frames
vp4/450... 32 frames
vp4/451... 33 frames
vp5/73... 34 frames
vp5/74... 34 frames
vp5/75... 31 frames
vp5/231... 33 frames
vp5/232... 34 frames
vp5/233... 35 frames
vp5/451... 32 frames
vp5/452... 32 frames
vp5/453... 34 frames
vp6/73... 34 frames
vp6/74... 34 frames
vp6/75... 31 frames
vp6/231... 33 frames
vp6/232... 34 fra